<a href="https://colab.research.google.com/github/karnaksp/ds_club/blob/main/notebooks/Interacting_with_CLIP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Interacting with CLIP

This is a self-contained notebook that shows how to download and run CLIP models, calculate the similarity between arbitrary image and text inputs, and perform zero-shot image classifications.

In [1]:
! pip install ftfy regex tqdm
! pip install git+https://github.com/openai/CLIP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-qk9tqn_p
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-qk9tqn_p
  Resolved https://github.com/openai/CLIP.git to commit ded190a052fdf4585bd685cee5bc96e0310d2c93
  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=102ea56ca99847ae69dcf25abf8bf485de1233abba2e8d11191384e61bf0c6c2
  Stored in directory: /tmp/pip-ephem-wheel-cache-tp5ha8w5/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [2]:
import numpy as np
import torch
from pkg_resources import packaging

print("Torch version:", torch.__version__)


Torch version: 2.10.0+cu128


/tmp/ipykernel_8525/189856270.py:3: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import packaging


In [3]:
import clip

clip.available_models()

['RN50',
 'RN101',
 'RN50x4',
 'RN50x16',
 'RN50x64',
 'ViT-B/32',
 'ViT-B/16',
 'ViT-L/14',
 'ViT-L/14@336px']

In [4]:
model, preprocess = clip.load("ViT-B/32")
model.cuda().eval()
input_resolution = model.visual.input_resolution
context_length = model.context_length
vocab_size = model.vocab_size

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Input resolution:", input_resolution)
print("Context length:", context_length)
print("Vocab size:", vocab_size)

100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 207MiB/s]


Model parameters: 151,277,313
Input resolution: 224
Context length: 77
Vocab size: 49408


In [6]:
import torch
from PIL import Image
import requests
from io import BytesIO
import pandas as pd
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

# 1. Список ваших ссылок (замените на свои или загрузите из csv)
urls = [
    "https://app-media.fitbod.me/v2/445/images/landscape/0_960x540.jpg",
    # Добавьте сюда ваши 1000 ссылок
]

# 2. Определение меток (классов)
descriptions = ["a photo of a black person", "a photo of a person who is not black"]
text_tokens = clip.tokenize(descriptions).to(device)

results = []

print("Начинаю разметку...")

for url in tqdm(urls):
    try:
        # Загружаем изображение по ссылке
        response = requests.get(url, timeout=10)
        image = Image.open(BytesIO(response.content)).convert("RGB")

        # Подготовка изображения для модели
        image_input = preprocess(image).unsqueeze(0).to(device)

        # Инференс (вычисление)
        with torch.no_grad():
            image_features = model.encode_image(image_input)
            text_features = model.encode_text(text_tokens)

            # Считаем сходство (косинусное)
            logits_per_image, _ = model(image_input, text_tokens)
            probs = logits_per_image.softmax(dim=-1).cpu().numpy()[0]

        # Определяем победивший класс
        is_black = "Yes" if probs[0] > probs[1] else "No"
        confidence = max(probs)

        results.append({
            "url": url,
            "is_black": is_black,
            "confidence": f"{confidence:.2%}"
        })

    except Exception as e:
        print(f"Ошибка с URL {url}: {e}")
        results.append({"url": url, "is_black": "Error", "confidence": "0%"})

# 3. Сохранение результата в CSV
df = pd.DataFrame(results)
df.to_csv("labeled_dataset.csv", index=False)
print("\nРазметка готова! Файл 'labeled_dataset.csv' сохранен.")

Начинаю разметку...


100%|██████████| 1/1 [00:01<00:00,  1.15s/it]


Разметка готова! Файл 'labeled_dataset.csv' сохранен.
